# Neural Networks (Cont.) + LSTM Network

This notebook attempts to implement the order of purchases as a feature in our models. We will attempt similar models as prior notebooks, but using the order of purchases by each user and survey response IDs (which uniquely identify each user) as features.

It also implements two attempts at creating a long short term memory (LSTM) network.

In [37]:
import pandas as pd
import numpy as np
import tensorflow as tf

We first need to add the survey response IDs and order date columns back into our data. Because we did not change the order of our data or expand any rows, we can simply add these features from the unencoded data to the encoded data, so we do not need to rerun all of our encoding steps.

In [38]:
data_unencoded = pd.read_csv('/workspaces/group-project-bas-team/data/data_unencoded.csv')
data_encoded = pd.read_csv('/workspaces/group-project-bas-team/data/cleaned_data.csv')

In [39]:
data_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 156 entries, Purchase Price Per Unit to life-changes_Moved place of residence,Had a child
dtypes: float64(142), int64(14)
memory usage: 186.9 MB


In [40]:
data_unencoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Order Date                157026 non-null  str    
 1   Purchase Price Per Unit   157026 non-null  float64
 2   Quantity                  157026 non-null  int64  
 3   Shipping Address State    157026 non-null  str    
 4   Title                     157026 non-null  str    
 5   ASIN/ISBN (Product Code)  157026 non-null  str    
 6   Category                  157026 non-null  str    
 7   Survey ResponseID         157026 non-null  str    
 8   age                       157026 non-null  str    
 9   hispanic                  157026 non-null  str    
 10  race                      157026 non-null  str    
 11  education                 157026 non-null  str    
 12  income                    157026 non-null  str    
 13  gender                    157026 non-null  str    
 14 

In [41]:
data_encoded[['Survey ResponseID', 'Order Date']] = data_unencoded[['Survey ResponseID', 'Order Date']]

/tmp/ipykernel_9064/601039707.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_encoded[['Survey ResponseID', 'Order Date']] = data_unencoded[['Survey ResponseID', 'Order Date']]


In [42]:
data_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 158 entries, Purchase Price Per Unit to Order Date
dtypes: float64(142), int64(14), str(2)
memory usage: 189.3 MB


In [43]:
# storing data_encoded as just data for easier code writing
data = data_encoded

In [44]:
# Ensuring the data is sorted by order date within each user's order history
data['Order Date'] = pd.to_datetime(data['Order Date'])
data = data.sort_values(by=['Survey ResponseID', 'Order Date'])

Next, we need to create a column that assigns a number to each order, representing the position in the sequence of a customer's orders.

In [45]:
data['Purchase_Order'] = data.groupby('Survey ResponseID')['Order Date'].rank(method='first').astype(int)

/tmp/ipykernel_9064/693081789.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Purchase_Order'] = data.groupby('Survey ResponseID')['Order Date'].rank(method='first').astype(int)


We will also add a column for time since last purchase

In [46]:
data['Days_Since_Last_Purchase'] = data.groupby('Survey ResponseID')['Order Date'].diff().dt.days.fillna(0)

/tmp/ipykernel_9064/646299313.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Days_Since_Last_Purchase'] = data.groupby('Survey ResponseID')['Order Date'].diff().dt.days.fillna(0)


In [47]:
data.head()

,Purchase Price Per Unit,Quantity,Title,ASIN/ISBN (Product Code),Category,age,hispanic,education,income,howmany,...,"life-changes_Lost a job ,Moved place of residence,Became pregnant","life-changes_Lost a job ,Moved place of residence,Became pregnant,Had a child","life-changes_Lost a job ,Moved place of residence,Had a child",life-changes_Moved place of residence,"life-changes_Moved place of residence,Became pregnant,Had a child","life-changes_Moved place of residence,Had a child",Survey ResponseID,Order Date,Purchase_Order,Days_Since_Last_Purchase
0,7.98,1,83780,34099,586,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-04,1,0.0
1,13.99,1,16614,43470,729,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-22,2,18.0
2,10.45,1,73826,47343,432,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-25,3,3.0
3,10.00,1,77034,21023,1240,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-25,4,0.0
4,10.99,1,62681,39476,331,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2019-02-18,5,55.0


In [48]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 160 entries, Purchase Price Per Unit to Days_Since_Last_Purchase
dtypes: datetime64[us](1), float64(143), int64(15), str(1)
memory usage: 191.7 MB


Next, we will remove the Order Date column, since we only needed it to order purchases by date and we already have columns for day/month/year.

In [49]:
data = data.drop(columns=['Order Date'])

Finally, we will encode the survey response ID column. Since there are thousands of unique IDs in our dataset, we will simply label encode this column.

The rationale for keeping this column is to distinguish multiple purchases that may have the same rank in purchase order (i.e., the first purchases made by different users will both be ranked 1). This is feasible because our training and testing sets are split by date, so each unique order ID will be present in both the training and testing sets.

In [50]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
data['Survey ResponseID'] = encoder.fit_transform(data['Survey ResponseID'])

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 159 entries, Purchase Price Per Unit to Days_Since_Last_Purchase
dtypes: float64(143), int64(16)
memory usage: 190.5 MB


Now we can attempt to build models using these new columns. We will start by replicating our MLP. The model will be modified slightly to account for the new temporal features added to the data.

In [51]:
from sklearn.preprocessing import StandardScaler

train = data[data['order_year']<=2021]
test = data[data['order_year']>2021]

train_cats = set(train['Category'].unique())
test_cats = set(test['Category'].unique())
unseen = test_cats - train_cats
if len(unseen) > 0:
    test = test[~test['Category'].isin(unseen)]

data_limited_train = train.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)
data_limited_test = test.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)

X_train = data_limited_train.drop('Category', axis=1)
y_train = data_limited_train['Category']

X_test = data_limited_test.drop('Category', axis=1)
y_test = data_limited_test['Category']

X_train = X_train.drop(X_train.filter(regex='^Shipping Address').columns, axis=1)
X_test = X_test.drop(X_test.filter(regex='^Shipping Address').columns, axis=1)

X_train = X_train.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])
X_test = X_test.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])

X_train_scaled = StandardScaler().fit_transform(X_train)
X_test_scaled = StandardScaler().fit_transform(X_test)

In [52]:
X_train_scaled.shape[1]

101

In [53]:
y_train.max()

np.int64(1624)

In [54]:
import tensorflow as tf

tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(101, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [55]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 300)            │        30,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 224,825 (878.22 KB)

 Trainable params: 224,825 (878.22 KB)

 Non-trainable params: 0 (0.00 B)

In [56]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])
              
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [57]:
history = model.fit(X_train_scaled, y_train, epochs=30, shuffle=False)

Epoch 1/30


W0000 00:00:1776717865.536188    9064 cpu_allocator_impl.cc:82] Allocation of 45916620 exceeds 10% of free system memory.


3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0622 - loss: 6.4134
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.0729 - loss: 6.0550
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0773 - loss: 5.9703
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0819 - loss: 5.8873
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0847 - loss: 5.8074
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0875 - loss: 5.7327
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0900 - loss: 5.6651
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.0928 - loss: 5.6045
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0948 - loss: 5.5500
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.0970 - loss: 5.5013
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0989 - loss: 5.4583
Epoch 12/30
3552/3552 ━━━━━━━━

In [58]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1355/1355 - 4s - 3ms/step - accuracy: 0.0708 - loss: 5.9342

Test accuracy: 0.07076092064380646


In [59]:
X_train_scaled.shape

(113655, 101)

In [60]:
X_train_reshaped = X_train_scaled.reshape((X_train_scaled.shape[0], 1, 101))
X_test_reshaped = X_test_scaled.reshape((X_test_scaled.shape[0], 1, 101))

In [61]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(1, 101))) 

model.add(tf.keras.layers.LSTM(150, return_sequences=False)) 

model.add(tf.keras.layers.Dense(1625, activation="softmax"))

In [62]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 150)            │       151,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1625)           │       245,375 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 396,575 (1.51 MB)

 Trainable params: 396,575 (1.51 MB)

 Non-trainable params: 0 (0.00 B)

In [63]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [64]:
history = model.fit(X_train_reshaped, y_train, epochs=30, shuffle=False)

Epoch 1/30


W0000 00:00:1776718259.407832    9064 cpu_allocator_impl.cc:82] Allocation of 45916620 exceeds 10% of free system memory.


3552/3552 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.0512 - loss: 7.2862
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.0555 - loss: 7.0820
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - accuracy: 0.0555 - loss: 6.9272
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - accuracy: 0.0576 - loss: 6.8167
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - accuracy: 0.0594 - loss: 6.7187
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - accuracy: 0.0614 - loss: 6.6275
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.0633 - loss: 6.5407
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.0658 - loss: 6.4592
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.0675 - loss: 6.3853
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - accuracy: 0.0686 - loss: 6.3207
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.0696 - loss: 6.2647
Epoch 12/30
3552/3552 ━━━━━━━━

In [65]:
test_loss, test_acc = model.evaluate(X_test_reshaped, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1355/1355 - 4s - 3ms/step - accuracy: 0.0577 - loss: 6.0838

Test accuracy: 0.05767499655485153


In [66]:
# Adding more layers

tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(1, 101))) 

model.add(tf.keras.layers.LSTM(150, return_sequences=True)) 
model.add(tf.keras.layers.LSTM(100, return_sequences=True)) 
model.add(tf.keras.layers.LSTM(100, return_sequences=False)) 

model.add(tf.keras.layers.Dense(1625, activation="softmax"))

In [67]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 1, 150)         │       151,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 1, 100)         │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100)            │        80,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 496,125 (1.89 MB)

 Trainable params: 496,125 (1.89 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

In [69]:
history = model.fit(X_train_reshaped, y_train, epochs=30, shuffle=False)

Epoch 1/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 25s 6ms/step - accuracy: 0.0552 - loss: 7.2891
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 25s 7ms/step - accuracy: 0.0555 - loss: 7.0963
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - accuracy: 0.0555 - loss: 6.9542
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 22s 6ms/step - accuracy: 0.0555 - loss: 6.8568
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 22s 6ms/step - accuracy: 0.0555 - loss: 6.7714
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - accuracy: 0.0555 - loss: 6.6911
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - accuracy: 0.0555 - loss: 6.6152
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 23s 6ms/step - accuracy: 0.0555 - loss: 6.5444
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 26s 7ms/step - accuracy: 0.0555 - loss: 6.4808
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 26s 7ms/step - accuracy: 0.0555 - loss: 6.4263
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 25s 7ms/step - accuracy: 0.0555 - loss: 6.3809
Epoch 12/30
3552/35

In [ ]:
test_loss, test_acc = model.evaluate(X_test_reshaped, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1355/1355 - 5s - 3ms/step - accuracy: 0.0396 - loss: 6.2906

Test accuracy: 0.03958088159561157


: 